# MultiHop-RAG corpus integrity

Deterministic checks against the authors’ pinned complete corpus. Run from the repository root; public originals and restored files are under `tmp/`. No model or API calls are made. Source availability does not prove semantic correctness of all answer labels. See `CORPUS-AUDIT.md` for the findings and provenance.

In [ ]:
import importlib.util
import json
from pathlib import Path

repository = Path.cwd()
if not (repository / "artifacts/search-recall/audit.py").exists():
    repository = repository.parents[1]
spec = importlib.util.spec_from_file_location(
    "corpus_audit", repository / "artifacts/search-recall/audit.py"
)
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
local = Path("/Users/pavlo_konovalov/repos/semgraph")
original = repository / "tmp/multihop-original"
load = lambda path: json.loads(path.read_text())

In [ ]:
result = audit.analyze(
    load(local / "sports.json"),
    load(local / "sports_q.json"),
    load(original / "corpus.json"),
    load(original / "MultiHopRAG.json"),
)
{
    key: result[key]
    for key in (
        "local",
        "original",
        "identical_articles",
        "identical_questions",
        "omitted_evidence_count",
        "restored_sports",
    )
}

In [ ]:
[
    {
        "question_index": row["local_index"],
        "original_evidence": row["original_evidence"],
        "local_evidence": row["local_evidence"],
        "missing_urls": [item["url"] for item in row["omitted_evidence"]],
    }
    for row in result["changed_questions"]
]

In [ ]:
articles, questions, manifest = audit.read_dataset(
    repository / "tmp/multihop-restored/sports-closed"
)
assert len(articles) == 253 and len(questions) == 609
assert audit.coverage(articles, questions)["evidence_count"] == 1410
manifest["origin"]